In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import os
import re
from scipy.interpolate import UnivariateSpline, PchipInterpolator
from sklearn.isotonic import isotonic_regression
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
from fun_LR_hydro_memory import *

# selected_lag_type_q = 'type1_incr1'

lag_ranges_q, widths_q, incr_name_q = make_lag_ranges(
    lag_increase=1,
    n_windows=6,
    incr_type=1
)
 
print('Streamflow lag ranges:\n', 
      lag_ranges_q)
 

Streamflow lag ranges:
 OrderedDict({'lag_0_0': (0, 0), 'lag_1_2': (1, 2), 'lag_3_5': (3, 5), 'lag_6_9': (6, 9), 'lag_10_14': (10, 14), 'lag_15_20': (15, 20)})


In [2]:
       
        
def compute_memory_from_spline_variants(df_config, perc_max_r2=0.9, spline_smoothing=0.5):
    """
    Compute memory estimates from a DataFrame built for a single well (well_data).

    This minimal, robust implementation collects available (pred_memory1..6, r2_pred1..6)
    pairs across the rows in `df_config`, filters out NaNs, and fits three curve variants
    (UnivariateSpline, PCHIP, and isotonic) to estimate the predictor memory at which
    R² first reaches `perc_max_r2` of the observed maximum R².

    Returns a dict with keys: 'lag_type' (None here), 'memory_spline','memory_pchip','memory_iso',
    'x_vals','y_spline','y_pchip','y_iso'.
    """
    # Collect x (memory) and y (r2) pairs from available columns
    x_list = []
    y_list = []
    for _, row in df_config.iterrows():
        for i in range(1, 7):
            xm = row.get(f'pred_memory{i}', np.nan)
            ym = row.get(f'r2_pred{i}', np.nan)
            x_list.append(float(xm))
            y_list.append(float(ym))

    if len(x_list) == 0:
        # nothing to fit
        return {
            'lag_type': None,
            'memory_spline': np.nan,
            'memory_pchip': np.nan,
            'memory_iso': np.nan,
            'x_vals': None,
            'y_spline': None,
            'y_pchip': None,
            'y_iso': None
        }

    x = np.array(x_list)
    y = np.array(y_list)

    # Remove NaNs and sort by x
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    if x.size == 0:
        return {
            'lag_type': None,
            'memory_spline': np.nan,
            'memory_pchip': np.nan,
            'memory_iso': np.nan,
            'x_vals': None,
            'y_spline': None,
            'y_pchip': None,
            'y_iso': None
        }

    idx = np.argsort(x)
    x_sorted = x[idx]
    y_sorted = y[idx]
    x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 500)
    r2_target = perc_max_r2 * np.nanmax(y_sorted)

    def first_cross(xvals, yvals, thr):
        above = yvals >= thr
        return xvals[np.argmax(above)] if np.any(above) else np.nan

    # Spline
    try:
        spline = UnivariateSpline(x_sorted, y_sorted, s=spline_smoothing)
        y_spline = spline(x_dense)
        mem_spline = first_cross(x_dense, y_spline, r2_target)
    except Exception:
        y_spline = None
        mem_spline = np.nan

    # PCHIP
    try:
        pchip = PchipInterpolator(x_sorted, y_sorted)
        y_pchip = pchip(x_dense)
        mem_pchip = first_cross(x_dense, y_pchip, r2_target)
    except Exception:
        y_pchip = None
        mem_pchip = np.nan

    # Isotonic
    try:
        # prefer the functional API if available
        try:
            y_iso = isotonic_regression(y_sorted)
        except Exception:
            # fallback to estimator API
            from sklearn.isotonic import IsotonicRegression
            ir = IsotonicRegression()
            # fit_transform expects X shape; use x_sorted as X to preserve monotonicity domain
            y_iso = ir.fit_transform(x_sorted, y_sorted)
        y_iso_dense = np.interp(x_dense, x_sorted, y_iso)
        mem_iso = first_cross(x_dense, y_iso_dense, r2_target)
    except Exception:
        y_iso_dense = None
        mem_iso = np.nan

    return {
        'lag_type': None,
        'memory_spline': mem_spline,
        'memory_pchip': mem_pchip,
        'memory_iso': mem_iso,
        'x_vals': x_dense,
        'y_spline': y_spline,
        'y_pchip': y_pchip,
        'y_iso': y_iso_dense,
    }


In [3]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.isotonic import IsotonicRegression

# =========================
# FUNCTION: isotonic memory
# =========================
def compute_memory_isotonic(df_config, perc_max_r2=0.9):
    # collect x (memory) and y (R2)
    x_list = []
    y_list = []

    for _, row in df_config.iterrows():
        for i in range(1, 7):
            xm = row.get(f'pred_memory{i}', np.nan)
            ym = row.get(f'r2_pred{i}', np.nan)
            x_list.append(float(xm))
            y_list.append(float(ym))

    x = np.array(x_list)
    y = np.array(y_list)

    # remove NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]

    if len(x) == 0:
        return {
            'memory_iso': np.nan,
            'x_vals': None,
            'y_iso': None
        }

    # sort by x
    idx = np.argsort(x)
    x_sorted = x[idx]
    y_sorted = y[idx]

    # isotonic fit
    ir = IsotonicRegression(increasing=True)
    y_iso = ir.fit_transform(x_sorted, y_sorted)

    # dense x for smooth plotting
    x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 500)
    y_iso_dense = np.interp(x_dense, x_sorted, y_iso)

    # threshold = 90% of max R2
    r2_target = perc_max_r2 * np.nanmax(y_sorted)

    # first x where fitted curve reaches threshold
    above = y_iso_dense >= r2_target
    if np.any(above):
        memory_iso = x_dense[np.argmax(above)]
    else:
        memory_iso = np.nan

    return {
        'memory_iso': memory_iso,
        'x_vals': x_dense,
        'y_iso': y_iso_dense
    }


# =========================
# LOAD R2 RESULTS
# =========================
df_q = pd.read_csv('../3_run_LR_Q/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"well_id": str})


In [4]:
######## Memory computation for STREAMFLOW (solo isotonic, sin filtro de R²) ########
# Requiere compute_memory_isotonic(df_config, perc_max_r2) definida antes

make_plots = True          # False: solo exporta el CSV (mucho más rápido)
selected_lag_type_q = 'type1_incr1'
perc_max_r2 = 0.9

# === Load R2 results ===
df_q = pd.read_csv('../3_run_LR_Q/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"gauge_id": str})
df = df_q[(df_q['target'] == 'q_an') & (df_q['predictor'] == 'pr_an')].copy()

# === Load predictions (used for time series plotting) ===
predictions_file = '../3_run_LR_Q/csv/q_sim_LR_trained_all.csv'
df_pred = pd.read_csv(predictions_file, parse_dates=['date'], dtype={"gauge_id": str})
df_pred = df_pred[(df_pred['target'] == 'q_an') & (df_pred['predictor'] == 'pr_an')]

# === Create pred_memory1..6 columns from lag_ranges ===
pattern = r"\(?\s*[-\d.]+\s*,\s*([-\d.]+)\s*\)?"
for i in range(1, 7):
    df[f'pred_memory{i}'] = np.nan

for idx in df.index:
    lr = df.at[idx, 'lag_ranges'] if 'lag_ranges' in df.columns else None
    if lr is None:
        vals = [np.nan] * 6
    else:
        matches = re.findall(pattern, str(lr))
        vals = []
        for i in range(6):
            if i < len(matches):
                try:
                    vals.append(float(matches[i]))
                except Exception:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)
    for j, v in enumerate(vals, start=1):
        df.at[idx, f'pred_memory{j}'] = v

if 'pred_memory' not in df.columns:
    df['pred_memory'] = df.apply(lambda r: np.array([r[f'pred_memory{i}'] for i in range(1, 7)]), axis=1)

xlim_min = df['pred_memory'].apply(lambda a: np.nanmin(a)).min() - 5
xlim_max = df['pred_memory'].apply(lambda a: np.nanmax(a)).max() + 5

# === Output ===
out_figs = 'figs_q'
os.makedirs(out_figs, exist_ok=True)
out_results = 'csv'
os.makedirs(out_results, exist_ok=True)

# === Colors ===
color_palette = plt.get_cmap('tab10')
lag_types_all = sorted(df['lag_type'].unique())
specified_colors = ['blue', 'purple', 'green']
model_colors = {}
for i, name in enumerate(lag_types_all):
    model_colors[name] = specified_colors[i] if i < len(specified_colors) else color_palette(i % 10)

memory_summary = []

# === Loop por cuenca (sin filtro de R²) ===
for well in sorted(df['gauge_id'].unique()):
    well_data = df[df['gauge_id'] == well]
    df_fit = well_data[well_data['lag_type'] == selected_lag_type_q]
    if df_fit.empty:
        print(f"Skipping basin {well}: no rows for {selected_lag_type_q}")
        continue

    max_r2_selected = df_fit['r2_all'].max()

    # --- memoria isotonic ---
    fit_results = compute_memory_isotonic(df_fit, perc_max_r2=perc_max_r2)
    memory_iso = fit_results['memory_iso']

    memory_summary.append({
        'gauge_id': well,
        'r2': max_r2_selected,
        'q_memory_iso': memory_iso
    })

    if not make_plots:
        continue

    fig = plt.figure(figsize=(13, 4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[3, 2])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Left plot: Observed vs predicted time series
    pred_data = df_pred[df_pred['gauge_id'] == well].copy()

    for name in sorted(well_data['lag_type'].unique()):
        pred_subset = pred_data[pred_data['lag_type'] == name]
        ax1.plot(pred_subset['date'], pred_subset['pred'],
                 label=name, color=model_colors.get(name),
                 linestyle='-', linewidth=1, zorder=1)

    obs = pred_data[pred_data['target'] == 'q_an']
    if not obs.empty and 'target_val' in obs.columns:
        ax1.scatter(obs['date'], obs['target_val'], color='black', s=6, zorder=2)
    ax1.set_title(f"Basin {well} – observed vs simulated Q, R² = {np.round(max_r2_selected, 2)}", loc='left')
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Q anom (mm)")
    ax1.grid(True, linestyle='--', alpha=0.5)
    handles, labels = ax1.get_legend_handles_labels()
    if labels:
        ax1.legend(fontsize=8)

    # Right plot: R² vs memory
    for name in sorted(well_data['lag_type'].unique()):
        pred_subset = well_data[well_data['lag_type'] == name]
        xm = np.array([pred_subset[f'pred_memory{i}'] for i in range(1, 7)])
        ym = np.array([pred_subset[f'r2_pred{i}'] for i in range(1, 7)])
        ax2.plot(xm, ym, label=name, marker='o', linestyle='None',
                 color=model_colors.get(name, 'red'), zorder=2)

    # isotonic fit only
    if fit_results['x_vals'] is not None and fit_results['y_iso'] is not None:
        ax2.plot(fit_results['x_vals'], fit_results['y_iso'],
                 '--', color='black', linewidth=1, label='Isotonic fit')
    if not np.isnan(memory_iso):
        ax2.axvline(memory_iso, color='black', linestyle='-', linewidth=1,
                    label=f'Q memory = {memory_iso:.0f} mo')

    ax2.set_title(f"R² with increasing # of predictors\nQ memory = {memory_iso:.0f} months", loc='left')
    ax2.set_xlabel("Pr_an lags (months)")
    ax2.set_ylabel("R²")
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(fontsize=8)
    ax2.set_xlim(xlim_min, xlim_max)

    plt.tight_layout()
    plt.savefig(os.path.join(out_figs, f"q_memory_{well}.png"), bbox_inches='tight')
    plt.close(fig)

# === Write summary (con atributos de las cuencas) ===
attrs_file = '../../data/camels/catchment_attributes.csv'
id_col_attrs = 'gauge_id'          # <- nombre de la columna de ID en catchment_attributes.csv

if memory_summary:
    df_memory_summary = pd.DataFrame(memory_summary)

    # leer el ID como texto para no perder ceros a la izquierda
    df_attrs = pd.read_csv(attrs_file, dtype={id_col_attrs: str})
    df_attrs = df_attrs.rename(columns={id_col_attrs: 'gauge_id'})
    df_attrs['gauge_id'] = df_attrs['gauge_id'].str.strip()
    df_memory_summary['gauge_id'] = df_memory_summary['gauge_id'].str.strip()

    # left join: se mantienen todas las cuencas con memoria, aunque no tengan atributos
    df_memory_summary = df_memory_summary.merge(df_attrs, on='gauge_id', how='left',
                                                suffixes=('', '_attr'))

    n_sin_attr = df_memory_summary[df_attrs.columns.drop('gauge_id')].isna().all(axis=1).sum()
    print(f"Cuencas sin atributos: {n_sin_attr}")

    out_csv = os.path.join(out_results, 'q_memory.csv')
    if os.path.exists(out_csv):
        os.remove(out_csv)
    df_memory_summary.to_csv(out_csv, index=False)
    # df_memory_summary.to_excel(os.path.join(out_results, 'q_memory.xlsx'), index=False)  # si lo querés en Excel
    print(f"Saved {len(df_memory_summary)} basins to {out_csv}")
else:
    print('No memory results to save')

Saved 484 basins to csv\q_memory.csv

LR models Finished! 
Finito! 
Terminado!

